# LangGraph G6 — Long-term memory
Rahul says "I prefer short bullet-point answers" on Monday. On Tuesday, in a *new* thread,
CampusAI should still know that. Thread memory cannot help: it is a different thread.

```text
STATE              = what is happening right now (this run)
SHORT-TERM MEMORY  = this conversation           (thread checkpoint, G5)
LONG-TERM MEMORY   = facts about a person        (store, keyed by user id, across threads)  <- this section
KNOWLEDGE          = documents anyone can read   (retrieval, G7)
```

LangGraph provides a **store**: key-value memory organised by namespace, such as
`("profiles", "rahul")`. Two mechanics carry it into the graph. **Runtime context** is data the
*application* passes into a run (who is talking, what role they have) that the model cannot
forge; nodes and tools read it from `runtime.context`. And `runtime.store` gives the same nodes
and tools the store. The agent decides *what* to remember; the application decides *where* it
goes and *who* can read it.

### Step 1 — Context schema, store, a profile-loading node and a remember tool

`load_profile` runs first on every turn and copies the user's stored facts into the state; the
agent node puts them into the system prompt. `remember_about_me` is a tool the model can call to
save a new fact; it reads the user id from the runtime, never from the model.

> **Why LangGraph has a *store* and *runtime context***
>
> Thread memory belongs to a conversation; some facts belong to a person and must outlive any thread, so the store is keyed by namespace instead of thread. Runtime context exists because the application, not the model, must be the source of identity: a model can be talked into claiming to be someone else, a value passed into invoke cannot.

In [ ]:
from langgraph.runtime import Runtime, get_runtime          # LangGraph: per-run context and store, from a node or a tool
from langgraph.store.memory import InMemoryStore            # LangGraph: long-term key-value memory

@dataclass
class Context:                                              # ours: what the application passes into each run
    user_id: str = "anonymous"
    role: str = "student"                                   # used by the permission section (G8)

class ProfileState(TypedDict, total=False):                 # ours
    messages: Annotated[list, add_messages]
    profile: list

@tool
def remember_about_me(fact: str) -> str:
    """Save a lasting fact or preference about the current user, e.g. how they like answers formatted."""
    runtime = get_runtime(Context)                          # LangGraph: the running graph's context and store
    namespace = ("profiles", runtime.context.user_id)
    existing = runtime.store.get(namespace, "facts")        # LangGraph store API: get(namespace, key)
    facts = (existing.value["facts"] if existing else []) + [fact]
    runtime.store.put(namespace, "facts", {"facts": facts}) # LangGraph store API: put(namespace, key, value)
    return f"Saved. I now know {len(facts)} fact(s) about you."

def load_profile(state: ProfileState, runtime: Runtime[Context]):   # ours: node; LangGraph injects the runtime
    existing = runtime.store.get(("profiles", runtime.context.user_id), "facts")
    return {"profile": existing.value["facts"] if existing else []}

def agent_with_profile(state: ProfileState):                # ours
    facts = "; ".join(state.get("profile") or []) or "none yet"
    persona = CAMPUS_PERSONA + f" Known facts about this user: {facts}\nFollow the user's stated preferences."
    reply = model.bind_tools(READ_TOOLS + [remember_about_me]).invoke([SystemMessage(persona)] + state["messages"])   # LangChain
    return {"messages": [reply]}

store = InMemoryStore()                                     # LangGraph
g = StateGraph(ProfileState, context_schema=Context)        # LangGraph: declare what context= must look like
g.add_node("load_profile", load_profile)
g.add_node("agent", agent_with_profile)
g.add_node("tools", ToolNode(READ_TOOLS + [remember_about_me]))
g.add_edge(START, "load_profile")
g.add_edge("load_profile", "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
campusai_v6 = g.compile(checkpointer=InMemorySaver(), store=store)   # LangGraph: persistence for threads AND a store for people
print("graph nodes:", [n for n in campusai_v6.get_graph().nodes if not n.startswith("__")])

> **What just happened**
>
> Definitions and a compile only. The graph now has `load_profile` before the agent and was compiled with both a checkpointer (threads) and a store (people). Nothing has been stored yet.

### Step 2 — Remember on one thread, recall on another, isolated per user

In [ ]:
monday = {"configurable": {"thread_id": "rahul-monday"}}
tuesday = {"configurable": {"thread_id": "rahul-tuesday"}}

out = campusai_v6.invoke({"messages": [HumanMessage("Please remember that I prefer short bullet-point answers.")]}, monday, context=Context(user_id="rahul"))   # LangGraph: context=
print("monday  :", text_of(out["messages"][-1])[:100])

out = campusai_v6.invoke({"messages": [HumanMessage("New conversation. What do you know about me and how should you answer?")]}, tuesday, context=Context(user_id="rahul"))
print("tuesday :", text_of(out["messages"][-1])[:120], "| profile loaded:", out["profile"])

out = campusai_v6.invoke({"messages": [HumanMessage("What do you know about me and how should you answer?")]}, {"configurable": {"thread_id": "priya-1"}}, context=Context(user_id="priya"))
print("priya   :", text_of(out["messages"][-1])[:100], "| profile loaded:", out["profile"])

print("\nstore contents:", [(item.namespace, item.value) for item in store.search(("profiles",))])   # LangGraph store API: search(namespace prefix)

> **What just happened**
>
> Monday: START -> `load_profile` found nothing for rahul -> `agent` requested `remember_about_me` -> `tools` ran it, and the tool wrote the fact under (profiles, rahul) using the user id from the runtime context, not from the model. Tuesday is a different thread with an empty message history, but `load_profile` read the store first; the profile line shows the fact arriving in the state before the model was called. Priya's run read a different namespace and found nothing.

### Recap

- **The problem we started with:** preferences vanished with the thread.
- **What we added:** a store keyed by user id, a runtime context set by the application, a profile-loading node and a remember tool.
- **What you saw in the output:** Tuesday's new thread recalled Monday's preference; another user's profile was empty.
- **Carry forward:** G7 gives the agent knowledge nobody typed into a prompt: documents retrieved by meaning.